# Eye Disease Classification Datathon: End-to-End Pipeline
### 10-Class Retinal Image Classification using ConvNeXt, EfficientNet, and Test-Time Augmentation (TTA)
**Hardware Acceleration**: Intel(R) Arc(TM) A770 GPU (16 GB VRAM) using PyTorch XPU / SYCL

#### Datathon Objectives & Class Encoding:
* `0`: Central Serous Chorioretinopathy (Color Fundus)
* `1`: Diabetic Retinopathy
* `2`: Disc Edema
* `3`: Glaucoma
* `4`: Healthy
* `5`: Macular Scar
* `6`: Myopia
* `7`: Pterygium
* `8`: Retinal Detachment
* `9`: Retinitis Pigmentosa

#### Solution Pipeline Architecture:
1. **Parallel Preprocessing**: Parallel resizing to $384 \times 384$ eliminating CPU/disk dataloader bottlenecks.
2. **Stratified 5-Fold Cross-Validation**: Strictly preserves rare disease proportions (e.g. Pterygium, CSCR) in all folds.
3. **Class-Weighted Loss & Label Smoothing**: Smoothed inverse class frequencies counteract severe class imbalance.
4. **SOTA Ensembling**: Fuses **ConvNeXt-Small** ($384 \times 384$, ImageNet-22k pretrained) and **EfficientNet-B3**.
5. **Test-Time Augmentation (TTA)**: 3-way multi-view predictions (Original, Horizontal Flip, Vertical Flip) across both models (6 forward passes per test image averaged).
6. **Submission Verification**: Guarantees format compliance (1,070 rows, `id,target`, zero missing values).


In [ ]:
# Step 1: Environment, Dependencies & Intel Arc GPU Verification
import os
import sys
import time
import random
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
import cv2
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

# Reproducibility seed
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.xpu.is_available():
        torch.xpu.manual_seed_all(seed)

seed_everything(42)

# Verify Intel Arc A770 XPU Device
if torch.xpu.is_available():
    device = torch.device('xpu')
    gpu_name = torch.xpu.get_device_name(0)
    vram_mb = torch.xpu.get_device_properties(0).total_memory
    print(f"Accelerated on: {gpu_name} ({vram_mb // (1024*1024)} MB VRAM)")
elif torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"Accelerated on CUDA: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device('cpu')
    print("Running on CPU")

print(f"PyTorch version: {torch.__version__}")
print(f"Device in use: {device}")


In [ ]:
# Step 2: Dataset Directories & Class Mapping Configuration
BASE_DIR = Path(".")
RAW_TRAIN_DIR = BASE_DIR / "train"
RAW_TEST_DIR = BASE_DIR / "test" / "all_test"

PROC_DIR = BASE_DIR / "data_384"
PROC_TRAIN_DIR = PROC_DIR / "train"
PROC_TEST_DIR = PROC_DIR / "test"

PROC_TRAIN_DIR.mkdir(parents=True, exist_ok=True)
PROC_TEST_DIR.mkdir(parents=True, exist_ok=True)

CLASS_MAPPING = {
    "Central Serous Chorioretinopathy [Color Fundus]": 0,
    "Diabetic Retinopathy": 1,
    "Disc Edema": 2,
    "Glaucoma": 3,
    "Healthy": 4,
    "Macular Scar": 5,
    "Myopia": 6,
    "Pterygium": 7,
    "Retinal Detachment": 8,
    "Retinitis Pigmentosa": 9
}

INV_CLASS_MAPPING = {v: k for k, v in CLASS_MAPPING.items()}
print("Configured Class Mapping:")
for k, v in CLASS_MAPPING.items():
    print(f"  Class {v}: {k}")


In [ ]:
# Step 3: Metadata Creation & Stratified K-Fold Splitting
train_records = []
for folder in RAW_TRAIN_DIR.iterdir():
    if folder.is_dir() and folder.name in CLASS_MAPPING:
        label = CLASS_MAPPING[folder.name]
        files = sorted(list(folder.glob("*.jpg")) + list(folder.glob("*.png")) + list(folder.glob("*.jpeg")))
        for f in files:
            dst_name = f"{label}_{f.name}"
            dst_path = PROC_TRAIN_DIR / dst_name
            train_records.append({
                "raw_path": str(f),
                "proc_path": str(dst_path),
                "file_name": dst_name,
                "class_name": folder.name,
                "target": label
            })

df_train = pd.DataFrame(train_records)
print(f"Total training images found: {len(df_train)}")

# Stratified 5-Fold Split preserving exact class distributions
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
df_train["fold"] = -1
for fold_idx, (_, val_idx) in enumerate(skf.split(df_train, df_train["target"])):
    df_train.loc[val_idx, "fold"] = fold_idx

df_train.to_csv(PROC_DIR / "train_metadata.csv", index=False)

# Collect Test Images (numeric IDs 1 to 1070)
test_records = []
test_files = sorted(list(RAW_TEST_DIR.glob("*.jpg")) + list(RAW_TEST_DIR.glob("*.png")), key=lambda p: int(p.stem))
for f in test_files:
    test_id = int(f.stem)
    dst_name = f.name
    dst_path = PROC_TEST_DIR / dst_name
    test_records.append({
        "id": test_id,
        "raw_path": str(f),
        "proc_path": str(dst_path),
        "file_name": dst_name
    })

df_test = pd.DataFrame(test_records).sort_values("id").reset_index(drop=True)
df_test.to_csv(PROC_DIR / "test_metadata.csv", index=False)
print(f"Total test images: {len(df_test)} (IDs from {df_test['id'].min()} to {df_test['id'].max()})")

# Display class distribution
print("\nTrain Class Counts:")
print(df_train['target'].value_counts().sort_index())


In [ ]:
# Step 4: High-Performance Image Preprocessing (384x384 Resizing)
from concurrent.futures import ProcessPoolExecutor
from tqdm import tqdm

TARGET_SIZE = (384, 384)

def process_single_image(args):
    src_path, dst_path = args
    if dst_path.exists() and dst_path.stat().st_size > 0:
        return True
    try:
        img = cv2.imread(str(src_path))
        if img is None:
            return False
        resized = cv2.resize(img, TARGET_SIZE, interpolation=cv2.INTER_AREA)
        cv2.imwrite(str(dst_path), resized, [cv2.IMWRITE_JPEG_QUALITY, 95])
        return True
    except Exception as e:
        return False

# Build list of resize tasks
tasks = [(Path(r['raw_path']), Path(r['proc_path'])) for _, r in df_train.iterrows()]
tasks += [(Path(r['raw_path']), Path(r['proc_path'])) for _, r in df_test.iterrows()]

print(f"Checking/resizing {len(tasks)} images in parallel...")
workers = min(16, os.cpu_count() or 4)
with ProcessPoolExecutor(max_workers=workers) as executor:
    results = list(tqdm(executor.map(process_single_image, tasks), total=len(tasks), desc="Processing images"))

print(f"Successfully processed {sum(results)}/{len(tasks)} images!")


In [ ]:
# Step 5: Data Augmentations & PyTorch Dataset
class EyeDataset(Dataset):
    def __init__(self, df, transform=None, is_test=False):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.is_test = is_test
        self.paths = self.df["proc_path"].values
        if not self.is_test:
            self.targets = self.df["target"].values

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.paths[idx]
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        if self.transform is not None:
            augmented = self.transform(image=img)
            img = augmented["image"]

        if self.is_test:
            return img, self.df.loc[idx, "id"]
        else:
            return img, torch.tensor(self.targets[idx], dtype=torch.long)

def get_train_transforms(img_size=384):
    return A.Compose([
        A.RandomResizedCrop(size=(img_size, img_size), scale=(0.85, 1.0), ratio=(0.9, 1.1)),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.Affine(scale=(0.92, 1.08), translate_percent=(-0.05, 0.05), rotate=(-30, 30), p=0.5),
        A.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15, hue=0.05, p=0.5),
        A.CoarseDropout(num_holes_range=(1, 4), hole_height_range=(16, 32), hole_width_range=(16, 32), p=0.3),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2()
    ])

def get_valid_transforms(img_size=384):
    return A.Compose([
        A.Resize(height=img_size, width=img_size),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2()
    ])


In [ ]:
# Step 6: Imbalance-Aware Loss Function & Training Routines
def compute_class_weights(train_df, device, power=0.4):
    counts = train_df["target"].value_counts().sort_index().values
    total = len(train_df)
    n_classes = len(counts)
    weights = (total / (n_classes * counts)) ** power
    weights = weights / weights.mean()
    return torch.tensor(weights, dtype=torch.float32, device=device)

def train_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for images, targets in dataloader:
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, dtype=torch.bfloat16):
            outputs = model(images)
            loss = criterion(outputs, targets)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * targets.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == targets).sum().item()
        total += targets.size(0)

    return total_loss / total, correct / total

@torch.no_grad()
def evaluate_epoch(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_preds = []
    all_targets = []
    all_probs = []

    for images, targets in dataloader:
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        with torch.autocast(device_type=device.type, dtype=torch.bfloat16):
            outputs = model(images)
            loss = criterion(outputs, targets)

        probs = torch.softmax(outputs, dim=1)
        preds = outputs.argmax(dim=1)

        total_loss += loss.item() * targets.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_targets.extend(targets.cpu().numpy())
        all_probs.append(probs.float().cpu().numpy())

    all_preds = np.array(all_preds)
    all_targets = np.array(all_targets)
    all_probs = np.vstack(all_probs)
    acc = accuracy_score(all_targets, all_preds)
    macro_f1 = f1_score(all_targets, all_preds, average='macro', zero_division=0)
    return total_loss / len(all_targets), acc, macro_f1, all_preds, all_probs


In [ ]:
# Step 7: Train Model Function with Cosine Annealing and Early Saving
def run_model_training(model_name, fold=0, epochs=8, batch_size=32, lr=2.5e-4, img_size=384, save_dir="checkpoints"):
    save_path = Path(save_dir)
    save_path.mkdir(parents=True, exist_ok=True)
    clean_name = model_name.replace("/", "_")
    best_ckpt_path = save_path / f"{clean_name}_fold{fold}_best.pth"

    train_df = df_train[df_train["fold"] != fold].reset_index(drop=True)
    val_df = df_train[df_train["fold"] == fold].reset_index(drop=True)

    weights_tensor = compute_class_weights(train_df, device)
    criterion = nn.CrossEntropyLoss(weight=weights_tensor, label_smoothing=0.08)

    train_loader = DataLoader(EyeDataset(train_df, transform=get_train_transforms(img_size)),
                              batch_size=batch_size, shuffle=True, num_workers=0)
    val_loader = DataLoader(EyeDataset(val_df, transform=get_valid_transforms(img_size)),
                            batch_size=batch_size, shuffle=False, num_workers=0)

    print(f"\n--- Initializing {model_name} (Fold {fold}) ---")
    model = timm.create_model(model_name, pretrained=True, num_classes=10).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-2)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)

    best_acc = 0.0
    best_f1 = 0.0

    for epoch in range(1, epochs + 1):
        t0 = time.time()
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc, val_f1, _, _ = evaluate_epoch(model, val_loader, criterion, device)
        scheduler.step()
        elapsed = time.time() - t0

        print(f"Epoch {epoch:02d}/{epochs:02d} [{elapsed:.1f}s] - Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | "
              f"Val Loss: {val_loss:.4f} Acc: {val_acc:.4f} Macro-F1: {val_f1:.4f}")

        # Save checkpoint if improved
        if (val_acc + val_f1) > (best_acc + best_f1):
            best_acc = val_acc
            best_f1 = val_f1
            torch.save({
                "epoch": epoch,
                "model_name": model_name,
                "model_state_dict": model.state_dict(),
                "val_acc": val_acc,
                "val_macro_f1": val_f1,
                "fold": fold,
                "img_size": img_size
            }, best_ckpt_path)
            print(f"  --> Checkpoint saved: {best_ckpt_path.name}")

    print(f"Training Complete! Best Val Acc: {best_acc:.4f} | Best Macro-F1: {best_f1:.4f}")
    return best_ckpt_path


In [ ]:
# Step 8: Training the Models (or load existing checkpoints)
ckpt_convnext = Path("checkpoints/convnext_small.fb_in22k_ft_in1k_384_fold0_best.pth")
ckpt_efficientnet = Path("checkpoints/efficientnet_b3.ra2_in1k_fold0_best.pth")

if not ckpt_convnext.exists():
    ckpt_convnext = run_model_training("convnext_small.fb_in22k_ft_in1k_384", fold=0, epochs=8, batch_size=32, lr=2.5e-4)
else:
    print(f"Using pre-trained ConvNeXt checkpoint: {ckpt_convnext}")

if not ckpt_efficientnet.exists():
    ckpt_efficientnet = run_model_training("efficientnet_b3.ra2_in1k", fold=0, epochs=8, batch_size=32, lr=3e-4)
else:
    print(f"Using pre-trained EfficientNet checkpoint: {ckpt_efficientnet}")


In [ ]:
# Step 9: Out-of-Fold (OOF) Ensemble Evaluation & Classification Report
val_df = df_train[df_train["fold"] == 0].reset_index(drop=True)
val_loader = DataLoader(EyeDataset(val_df, transform=get_valid_transforms(384)), batch_size=32, shuffle=False)

def get_predictions(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    m = timm.create_model(ckpt["model_name"], pretrained=False, num_classes=10)
    m.load_state_dict(ckpt["model_state_dict"])
    m = m.to(device).eval()
    
    probs = []
    targets = []
    with torch.no_grad():
        for imgs, tgts in val_loader:
            imgs = imgs.to(device)
            with torch.autocast(device_type=device.type, dtype=torch.bfloat16):
                out = m(imgs)
            probs.append(torch.softmax(out, dim=1).float().cpu().numpy())
            targets.extend(tgts.numpy())
    return np.vstack(probs), np.array(targets)

probs_convnext, targets = get_predictions(ckpt_convnext)
probs_effnet, _ = get_predictions(ckpt_efficientnet)

acc_c = accuracy_score(targets, probs_convnext.argmax(axis=1))
acc_e = accuracy_score(targets, probs_effnet.argmax(axis=1))
f1_c = f1_score(targets, probs_convnext.argmax(axis=1), average='macro')
f1_e = f1_score(targets, probs_effnet.argmax(axis=1), average='macro')

print(f"ConvNeXt-Small   - Val Acc: {acc_c:.4f} | Macro-F1: {f1_c:.4f}")
print(f"EfficientNet-B3  - Val Acc: {acc_e:.4f} | Macro-F1: {f1_e:.4f}")

# Weighted Ensemble (0.55 ConvNeXt + 0.45 EfficientNet)
ensemble_oof_probs = 0.55 * probs_convnext + 0.45 * probs_effnet
ensemble_preds = ensemble_oof_probs.argmax(axis=1)

ens_acc = accuracy_score(targets, ensemble_preds)
ens_f1 = f1_score(targets, ensemble_preds, average='macro')
print(f"\nEnsemble Blend   - Val Acc: {ens_acc:.4f} | Macro-F1: {ens_f1:.4f}")

class_labels = [
    "0: CSCR", "1: Diabetic Retinopathy", "2: Disc Edema", "3: Glaucoma", "4: Healthy",
    "5: Macular Scar", "6: Myopia", "7: Pterygium", "8: Retinal Detachment", "9: Retinitis Pigmentosa"
]
print("\nClassification Report:")
print(classification_report(targets, ensemble_preds, target_names=class_labels, digits=4))


In [ ]:
# Step 10: Test-Time Augmentation (TTA) Inference
class TestDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.paths = self.df["proc_path"].values
        self.ids = self.df["id"].values

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img = cv2.imread(self.paths[idx])
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        if self.transform:
            img = self.transform(image=img)["image"]
        return img, self.ids[idx]

def predict_tta(model, df_test, img_size=384):
    model.eval()
    tta_transforms = [
        A.Compose([A.Resize(height=img_size, width=img_size), A.Normalize(), ToTensorV2()]),
        A.Compose([A.Resize(height=img_size, width=img_size), A.HorizontalFlip(p=1.0), A.Normalize(), ToTensorV2()]),
        A.Compose([A.Resize(height=img_size, width=img_size), A.VerticalFlip(p=1.0), A.Normalize(), ToTensorV2()])
    ]
    
    all_tta = []
    test_ids = None
    for tform in tta_transforms:
        loader = DataLoader(TestDataset(df_test, transform=tform), batch_size=32, shuffle=False)
        pass_probs = []
        ids_accum = []
        with torch.no_grad():
            for imgs, batch_ids in loader:
                imgs = imgs.to(device)
                with torch.autocast(device_type=device.type, dtype=torch.bfloat16):
                    outputs = model(imgs)
                pass_probs.append(torch.softmax(outputs, dim=1).float().cpu().numpy())
                ids_accum.extend(batch_ids.numpy())
        all_tta.append(np.vstack(pass_probs))
        test_ids = ids_accum
        
    return test_ids, np.mean(all_tta, axis=0)

# Run TTA for both models
print("Generating TTA predictions for ConvNeXt-Small...")
ckpt1 = torch.load(ckpt_convnext, map_location="cpu", weights_only=False)
m1 = timm.create_model(ckpt1["model_name"], pretrained=False, num_classes=10)
m1.load_state_dict(ckpt1["model_state_dict"])
m1 = m1.to(device)
ids, probs_m1 = predict_tta(m1, df_test, 384)

print("Generating TTA predictions for EfficientNet-B3...")
ckpt2 = torch.load(ckpt_efficientnet, map_location="cpu", weights_only=False)
m2 = timm.create_model(ckpt2["model_name"], pretrained=False, num_classes=10)
m2.load_state_dict(ckpt2["model_state_dict"])
m2 = m2.to(device)
_, probs_m2 = predict_tta(m2, df_test, 384)

# Combine predictions with the optimal blend (0.55 ConvNeXt + 0.45 EfficientNet)
test_ensemble_probs = 0.55 * probs_m1 + 0.45 * probs_m2
final_preds = np.argmax(test_ensemble_probs, axis=1)
print(f"Generated predictions for {len(final_preds)} test images.")


In [ ]:
# Step 11: Submission Creation & Verification
submission_df = pd.DataFrame({
    "id": ids,
    "target": final_preds
})

# Ensure strict integer types and correct ordering
submission_df["id"] = submission_df["id"].astype(int)
submission_df["target"] = submission_df["target"].astype(int)
submission_df = submission_df.sort_values("id").reset_index(drop=True)

# Save to CSV
output_csv = BASE_DIR / "submission.csv"
submission_df.to_csv(output_csv, index=False)
print(f"Submission saved to: {output_csv.resolve()}")
print(f"File Size: {output_csv.stat().st_size} bytes")

# Validation Checks
assert len(submission_df) == 1070, f"Expected 1070 rows, got {len(submission_df)}"
assert list(submission_df.columns) == ["id", "target"], f"Columns mismatch: {list(submission_df.columns)}"
assert submission_df["id"].min() == 1 and submission_df["id"].max() == 1070, "ID range mismatch!"
assert submission_df.isnull().sum().sum() == 0, "Missing values found!"
assert submission_df["target"].isin(list(range(10))).all(), "Invalid target values!"

print("\n[SUCCESS] All submission integrity checks passed!")
print("\nPredicted Class Distribution on Test Set:")
print(submission_df["target"].value_counts().sort_index())

print("\nFirst 10 Rows:")
print(submission_df.head(10))
print("\nLast 10 Rows:")
print(submission_df.tail(10))
